# 05 - Run 8: IR + Depth Fusion
Dual-stream fusion: fine-tune separate CNNs for IR and Depth,
extract features from both, concatenate (1024-dim), train LSTM.

**Pipeline:** Verify depth data → Compute stats → Fine-tune depth CNN → Extract depth features → Merge (k=2) → Verify alignment → Train fusion LSTM

**Runtime:** ~60-75 min total on Colab T4.

In [ ]:
# Colab Setup
import os
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/Driver-Activity-Recognition'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/batuhne/Driver-Activity-Recognition.git {REPO_DIR}

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    DATA_ROOT = '/content/drive/MyDrive/DriveAndAct'
else:
    DATA_ROOT = './data'

print(f'Working directory: {os.getcwd()}')
print(f'Data root: {DATA_ROOT}')

In [ ]:
# Verify depth videos exist
from pathlib import Path

depth_dir = os.path.join(DATA_ROOT, 'kinect_depth_mp4')
for vp in range(1, 16):
    vp_dir = os.path.join(depth_dir, f'vp{vp}')
    videos = list(Path(vp_dir).glob('*.mp4'))
    assert len(videos) == 2, f"vp{vp}: expected 2 videos, found {len(videos)}"
    for v in videos:
        print(f"  {v.name} ({v.stat().st_size / 1024**2:.0f} MB)")
print(f"\nAll 30 depth videos verified.")

In [ ]:
# Depth normalization statistics (pre-computed)
depth_mean = 0.2641
depth_std = 0.1505
print(f"Depth mean={depth_mean}, std={depth_std}")

In [ ]:
# Backup IR checkpoint + Depth CNN fine-tuning
import shutil
import torch
from src.cnn_finetune import finetune_cnn

config = load_config()
config['data']['root'] = DATA_ROOT
config['data']['annotation_dir'] = 'activities_3s/kinect_depth'
config['data']['video_dir'] = 'kinect_depth_mp4'
config['processing']['ir_mean'] = depth_mean
config['processing']['ir_std'] = depth_std

drive_output = os.path.join(DATA_ROOT, 'results')
config['output']['checkpoint_dir'] = os.path.join(drive_output, 'checkpoints')
config['output']['log_dir'] = os.path.join(drive_output, 'logs')

config['model']['freeze_mode'] = 'layer4'

config['cnn_finetune'] = {
    'batch_size': 64,
    'epochs': 15,
    'early_stop_patience': 7,
    'cnn_lr': 1e-4,
    'fc_lr': 1e-3,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'gradient_clip': 1.0,
    'use_amp': True,
    'use_weighted_sampler': True,
}
config['training']['en_beta'] = 0.99
config['training']['num_workers'] = 2

checkpoint_dir = config['output']['checkpoint_dir']

# Automatic IR checkpoint backup
ir_ckpt = os.path.join(checkpoint_dir, 'cnn_finetuned.pth')
ir_backup = os.path.join(checkpoint_dir, 'cnn_finetuned_ir_backup.pth')
if os.path.exists(ir_ckpt) and not os.path.exists(ir_backup):
    shutil.copy2(ir_ckpt, ir_backup)
    print(f"IR checkpoint backed up: {ir_backup}")

print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f"Depth mean={config['processing']['ir_mean']}, std={config['processing']['ir_std']}")
print(f"freeze_mode: {config['model']['freeze_mode']}")
print(f"CNN LR: {config['cnn_finetune']['cnn_lr']}, FC LR: {config['cnn_finetune']['fc_lr']}")

# Run depth fine-tuning (resumes from cnn_finetune_last.pth if exists)
best_path = finetune_cnn(config)

# Save depth checkpoint with distinct name
depth_ckpt = os.path.join(checkpoint_dir, 'cnn_finetuned_depth.pth')
shutil.copy2(best_path, depth_ckpt)
print(f"\nDepth CNN saved to: {depth_ckpt}")

In [ ]:
# Restore IR checkpoint
ir_ckpt = os.path.join(checkpoint_dir, 'cnn_finetuned.pth')
ir_backup = os.path.join(checkpoint_dir, 'cnn_finetuned_ir_backup.pth')
if os.path.exists(ir_backup):
    shutil.copy2(ir_backup, ir_ckpt)
    print(f"IR checkpoint restored: {ir_ckpt}")

# Verify both checkpoints exist
for name in ['cnn_finetuned.pth', 'cnn_finetuned_depth.pth', 'cnn_finetuned_ir_backup.pth']:
    p = os.path.join(checkpoint_dir, name)
    exists = os.path.exists(p)
    size = os.path.getsize(p) / 1024**2 if exists else 0
    print(f"  {'OK' if exists else 'MISSING'}: {name} ({size:.1f} MB)")

In [ ]:
# Depth feature extraction
from src.feature_extract import extract_features

config = load_config()
config['data']['root'] = DATA_ROOT
config['data']['annotation_dir'] = 'activities_3s/kinect_depth'
config['data']['video_dir'] = 'kinect_depth_mp4'
config['processing']['ir_mean'] = round(depth_mean, 4)
config['processing']['ir_std'] = round(depth_std, 4)
config['features']['save_dir'] = 'features_depth_finetuned'
config['features']['dtype'] = 'float32'

depth_ckpt = os.path.join(DATA_ROOT, 'results', 'checkpoints', 'cnn_finetuned_depth.pth')
print(f'Depth CNN checkpoint: {depth_ckpt}')
print(f'Output dir: {os.path.join(DATA_ROOT, config["features"]["save_dir"])}')

extract_features(config, cnn_checkpoint=depth_ckpt)

In [ ]:
# Depth feature merging (k=2)
from src.merge_features import merge_features
import logging
logging.basicConfig(level=logging.INFO)

source = os.path.join(DATA_ROOT, 'features_depth_finetuned')
output = os.path.join(DATA_ROOT, 'features_depth_merged_k2')
merge_features(source, output, merge_k=2, pad_mode='repeat')

In [ ]:
# Verify feature alignment between IR and Depth
import pandas as pd
import numpy as np

ir_dir = os.path.join(DATA_ROOT, 'features_merged_k2')
depth_dir = os.path.join(DATA_ROOT, 'features_depth_merged_k2')

for split in ['train', 'val', 'test']:
    ir_manifest = pd.read_csv(os.path.join(ir_dir, split, 'manifest.csv'))
    depth_manifest = pd.read_csv(os.path.join(depth_dir, split, 'manifest.csv'))

    assert len(ir_manifest) == len(depth_manifest), \
        f"{split}: IR has {len(ir_manifest)} segments, depth has {len(depth_manifest)}"
    assert (ir_manifest['label'] == depth_manifest['label']).all(), \
        f"{split}: label mismatch between IR and depth"

    # Check feature shapes
    ir_sample = np.load(os.path.join(ir_dir, split, ir_manifest.iloc[0]['filename']))
    depth_sample = np.load(os.path.join(depth_dir, split, depth_manifest.iloc[0]['filename']))
    print(f"{split}: {len(ir_manifest)} segments, IR shape={ir_sample.shape}, Depth shape={depth_sample.shape}")

print("\nAlignment verified: IR and Depth features match perfectly.")

In [ ]:
# Fusion LSTM training
import torch, shutil
from src.utils import load_config
from src.train import train

config = load_config()
config['data']['root'] = DATA_ROOT

drive_output = os.path.join(DATA_ROOT, 'results')
config['output']['checkpoint_dir'] = os.path.join(drive_output, 'checkpoints')
config['output']['log_dir'] = os.path.join(drive_output, 'logs')
config['output']['figure_dir'] = os.path.join(drive_output, 'figures')

# === FUSION CONFIG ===
config['features']['save_dir'] = 'features_merged_k2'             # IR features (primary)
config['features']['fusion_dir'] = 'features_depth_merged_k2'     # Depth features (secondary)
config['model']['feature_dim'] = 1024                              # 512 IR + 512 Depth
config['training']['mode'] = 'feature_based'

# Model config: same as Run 7b k=2 (isolate fusion variable)
config['model']['use_layernorm'] = True
config['model']['bidirectional'] = True
config['model']['pooling'] = 'attention'
config['model']['lstm_hidden'] = 256
config['model']['lstm_dropout'] = 0.3

# Training config
config['training']['batch_size'] = 32
config['training']['lr'] = 0.001
config['training']['loss_type'] = 'ce'
config['training']['label_smoothing'] = 0.1
config['training']['mixup_alpha'] = 0.0
config['training']['noise_std'] = 0.0
config['training']['weight_decay'] = 0.0001
config['training']['epochs'] = 50
config['training']['early_stop_patience'] = 12
config['training']['gradient_clip'] = 1.0
config['training']['use_weighted_sampler'] = True
config['training']['en_beta'] = 0.99
config['training']['num_workers'] = 2

# Delete old checkpoints
for f in ['last_checkpoint.pth', 'best_model.pth']:
    p = os.path.join(config['output']['checkpoint_dir'], f)
    if os.path.exists(p):
        os.remove(p)
        print(f"Deleted: {p}")

if os.path.exists(config['output']['log_dir']):
    shutil.rmtree(config['output']['log_dir'])

print(f"IR features: {config['features']['save_dir']}")
print(f"Depth features: {config['features']['fusion_dir']}")
print(f"Feature dim: {config['model']['feature_dim']}")
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

model = train(config)

## Monitor Training
Launch TensorBoard to monitor training progress in real-time.

In [ ]:
# TensorBoard (works in Colab and Jupyter)
%load_ext tensorboard
%tensorboard --logdir {config['output']['log_dir']}